# Pydantic Data Validation Example

In [33]:
from pydantic import BaseModel

# Validate the User Data
class User(BaseModel):
    id:int
    name:str
    
user = User(id=1, name="John")
print(user)


id=1 name='John'


In [34]:
# Use the Default Values
class User(BaseModel):
    id:int
    name:str
    signup_ts:str | None = None
    salary: float | None = None

user1 = User(id=2, name="Smith", salary=80000)
print(user1)
    

id=2 name='Smith' signup_ts=None salary=80000.0


In [35]:
from dataclasses import dataclass

@dataclass
class User():
    id:int
    name:str
    signup_ts: str | None = None
    salary: float | None = None
    
user2 = User(id='2', name="Peter", salary=90000)
print(user2)

User(id='2', name='Peter', signup_ts=None, salary=90000)


# Scenario: User Registration API Input

In [36]:
# You're building a backend service that receives JSON payloads from frontend form:
{
    "username": "Johndoe",
    "password": "secret123",
    "email": "john10@gmail.com",
    "age": "30",
    "join_date":"2025-01-01"
    
}


{'username': 'Johndoe',
 'password': 'secret123',
 'email': 'john10@gmail.com',
 'age': '30',
 'join_date': '2025-01-01'}

You need to:
- Validate the email format

- Ensure age is a positive integer

- Ensure password meets minimum length requirements

- Convert "30" to integer

- Parse "2025-01-01" into a datetime.date

In [37]:
from pydantic import BaseModel, EmailStr, field_validator, ValidationError
from datetime import date

class UserRegistration(BaseModel):
    username:str
    password:str
    email:EmailStr
    age: int
    join_date: date
    
    @field_validator("age", mode='after')
    @classmethod
    def validate_age(cls,value):
        if value < 0:
            raise ValueError("Age must be positive")
        return value
    
    @field_validator("password", mode='after')
    @classmethod
    def validate_password(cls,value):
        if len(value) < 10:
            raise ValueError("Password must be at least 10 charaters long")
        return value

In [38]:
# Simulate incoming JSON
incoming_data = {
    "username": "Johndoe",
    "password": "secret12334",
    "email": "john10@gmail.com",
    "age": "30",
    "join_date":"2025-01-01"
}

try:
    user3 = UserRegistration(**incoming_data)
    print("JSON output:", user3.model_dump_json())
except ValidationError as e:
    print("❌ Validation Error:")
    print(e)

JSON output: {"username":"Johndoe","password":"secret12334","email":"john10@gmail.com","age":30,"join_date":"2025-01-01"}


# Nested Models

In [39]:
class Address(BaseModel):
    street: str
    city:str
    zip_code: str
    
class UserRegistrationWithAddress(UserRegistration):
    address: Address

incoming_data = {
    "username": "johndoe",
    "password": "secret12345",
    "email": "john@example.com",
    "age": "30",
    "join_date": "2025-01-01",
    "address": {
        "street": "123 Main St",
        "city": "New York",
        "zip_code": "10001"
    }
}

try:
    userWithAdd = UserRegistrationWithAddress(**incoming_data)
    print("JSON output:", userWithAdd.model_dump_json())
except ValidationError as e:
    print("❌ Validation Error:")
    print(e)

JSON output: {"username":"johndoe","password":"secret12345","email":"john@example.com","age":30,"join_date":"2025-01-01","address":{"street":"123 Main St","city":"New York","zip_code":"10001"}}
